# 911 Emergency Calls: Analysis Walkthrough
**Question:** how should a dispatch center staff and position resources given *when*, *where* and *what kind* of calls arrive?

This notebook narrates the analysis; all logic lives in `src/` so it is tested and reusable. Replace each *Takeaway* with your own interpretation after running on the real data.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import anomalies, data, eda, forecast, geo, quality, staffing, stats_tests
from src.config import DATA_RAW, FIGURES

DATA = DATA_RAW if DATA_RAW.exists() else ROOT / "data" / "911_sample.csv"  # sample = SYNTHETIC
print("Using:", DATA)

## 1. Data quality audit
Audit before analysis: duplicates, bad timestamps, GPS outliers, missing days.

In [ ]:
raw = data.load_raw(DATA)
df = data.clean(raw)
quality.quality_report(raw, df)

**Takeaway:** _what share of rows was removed, and are there gaps in time coverage that could distort trends?_

## 2. When and what: exploratory analysis

In [ ]:
eda.run_all(df)
display(Image(str(FIGURES / "03_heatmap_all.png")))
display(Image(str(FIGURES / "04_monthly_trend.png")))

**Takeaway:** _peak hours, weekday pattern, seasonality, category mix._

## 3. Is the pattern real? Statistical tests with effect sizes

In [ ]:
display(pd.Series(stats_tests.weekday_vs_weekend(df)))
display(pd.Series(stats_tests.category_mix_by_township(df)))

**Takeaway:** _with this many rows p-values are tiny, so interpret the effect sizes (rank-biserial, Cohen's d, Cramér's V)._

## 4. Where: hotspots and station coverage

In [ ]:
hotspots = geo.find_hotspots(df)
display(hotspots.head(10))
curve, centers = geo.station_coverage_curve(df)
display(Image(str(FIGURES / "06_station_coverage_curve.png")))
geo.hotspot_stability(df)

**Takeaway:** _where do calls cluster, where do extra stations stop paying off, and do hotspots persist year to year?_

## 5. Forecasting with honest validation
Seasonal-naive baseline vs gradient boosting, chronological hold-out, 80% prediction interval, feature ablation.

In [ ]:
holdout, cv, interval = forecast.evaluate(df)
display(holdout.round(3))
display(cv.round(3))
print(interval)
from src.config import REPORTS

display(pd.read_csv(REPORTS / "forecast_ablation.csv"))
display(Image(str(FIGURES / "07_forecast_holdout.png")))

**Takeaway:** _does the model beat the baseline in every fold? is the interval calibrated? which feature group matters most?_

## 6. Anomalous days

In [ ]:
daily = anomalies.detect_daily_anomalies(df)
anomalies.save_anomalies(daily)
display(daily[daily["anomaly"]].head(15))
display(Image(str(FIGURES / "09_anomalies.png")))

**Takeaway:** _which flagged days can you explain (holidays, storms, outages, reporting changes)?_

## 7. Decision: staffing recommendation
Units = ceil(p90 demand × handling hours ÷ target utilization). Handling time and utilization are **assumptions**.

In [ ]:
staff = staffing.staffing_table(df)
staffing.plot_staffing_heatmap(staff, "EMS")
display(Image(str(FIGURES / "08_staffing_ems.png")))

**Recommendation:** _summarise in 3 bullets: staffing shape, station placement, and the main caveat._